In [5]:
import pypsa
import pandas as pd

n = pypsa.Network()
n.set_snapshots(["now"])

n.add("Bus", "Bus 1")
n.add("Bus", "Bus 2")
n.add("Bus", "Bus 3")

# Γραμμές με μεγάλη χωρητικότητα για να μην έχουμε σφάλμα "Infeasible"
n.add("Line", "Line 1-2", bus0="Bus 1", bus1="Bus 2", x=0.1, s_nom=1000)
n.add("Line", "Line 2-3", bus0="Bus 2", bus1="Bus 3", x=0.1, s_nom=1000)

n.add("Load", "Consumer", bus="Bus 3", p_set=150)

n.add("Generator", "Cheap_Coal", bus="Bus 1", p_nom=200, marginal_cost=20)
n.add("Generator", "Expensive_Gas", bus="Bus 2", p_nom=200, marginal_cost=50)

# Επίλυση - Αποθηκεύουμε το αποτέλεσμα της κατάστασης
status, message = n.optimize(solver_name='highs')

print(f"Solver Status: {status}")

if status == "ok":
    # Χρήση του .loc[snapshot] για ασφάλεια
    res_coal = n.generators_t.p.loc["now", "Cheap_Coal"]
    res_gas = n.generators_t.p.loc["now", "Expensive_Gas"]
    print(f"Παραγωγή Coal: {res_coal} MW")
    print(f"Παραγωγή Gas: {res_gas} MW")
else:
    print("Η βελτιστοποίηση απέτυχε! Δοκίμασε να ελέγξεις αν ο solver highs είναι σωστά εγκατεστημένος.")
    # Εκτύπωση για να δούμε τι έχει μέσα το αντικείμενο αν αποτύχει
    print("Διαθέσιμες στήλες αποτελεσμάτων:", n.generators_t.p.columns)

    

Index(['Bus 1', 'Bus 2', 'Bus 3'], dtype='str', name='name')
Index(['Line 1-2', 'Line 2-3'], dtype='str', name='name')
Index(['Line 1-2', 'Line 2-3'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.05s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 11 duals
Objective: 3.00e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


Solver Status: ok
Παραγωγή Coal: 150.0 MW
Παραγωγή Gas: -0.0 MW


In [6]:
import pypsa
import pandas as pd

n = pypsa.Network()
n.set_snapshots(["now"])

# Δίκτυο
n.add("Bus", "Bus 1")
n.add("Bus", "Bus 2")
n.add("Bus", "Bus 3")

# Γραμμές (Περιορίζουμε τη Line 1-2 στα 60MW για να δούμε "congestion")
n.add("Line", "Line 1-2", bus0="Bus 1", bus1="Bus 2", x=0.1, s_nom=60)
n.add("Line", "Line 2-3", bus0="Bus 2", bus1="Bus 3", x=0.1, s_nom=1000)

n.add("Load", "Consumer", bus="Bus 3", p_set=150)

n.add("Generator", "Cheap_Coal", bus="Bus 1", p_nom=200, marginal_cost=20)
n.add("Generator", "Expensive_Gas", bus="Bus 2", p_nom=200, marginal_cost=50)

# --- ΕΚΤΕΛΕΣΗ ΚΑΙ ΑΠΟΚΡΙΣΗ SOLVER ---
status, message = n.optimize(solver_name='highs')

print("-" * 30)
print(f"SOLVER STATUS: {status}")
print(f"SOLVER MESSAGE: {message}")
print("-" * 30)

if status == "ok":
    # 1. Παραγωγή ανά μονάδα
    print("ΑΠΟΤΕΛΕΣΜΑΤΑ ΠΑΡΑΓΩΓΗΣ:")
    print(n.generators_t.p.loc["now"])
    print("-" * 10)

    # 2. Ροές στις γραμμές
    print("ΡΟΕΣ ΓΡΑΜΜΩΝ (MW):")
    print(n.lines_t.p0.loc["now"])
    print("-" * 10)

    # 3. Οριακές Τιμές Κόμβων (Shadow Prices / LMPs)
    # Δείχνει πόσο θα κόστιζε 1 επιπλέον MW σε κάθε σημείο
    print("ΤΙΜΕΣ ΕΝΕΡΓΕΙΑΣ ΑΝΑ ΚΟΜΒΟ (€/MWh):")
    print(n.buses_t.marginal_price.loc["now"])
    
else:
    print("Πρόβλημα στην επίλυση.")

Index(['Bus 1', 'Bus 2', 'Bus 3'], dtype='str', name='name')
Index(['Line 1-2', 'Line 2-3'], dtype='str', name='name')
Index(['Line 1-2', 'Line 2-3'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.03s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 11 duals
Objective: 5.70e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


------------------------------
SOLVER STATUS: ok
SOLVER MESSAGE: optimal
------------------------------
ΑΠΟΤΕΛΕΣΜΑΤΑ ΠΑΡΑΓΩΓΗΣ:
name
Cheap_Coal       60.0
Expensive_Gas    90.0
Name: now, dtype: float64
----------
ΡΟΕΣ ΓΡΑΜΜΩΝ (MW):
name
Line 1-2     60.0
Line 2-3    150.0
Name: now, dtype: float64
----------
ΤΙΜΕΣ ΕΝΕΡΓΕΙΑΣ ΑΝΑ ΚΟΜΒΟ (€/MWh):
name
Bus 1    20.0
Bus 2    50.0
Bus 3    50.0
Name: now, dtype: float64
